# Extracting Edge Attributes from TRAPI Results

This notebook demonstrates how to extract structured metadata (publications, supporting text, confidence scores) from TRAPI edge attributes.

In [ ]:
from TCT.attribute_extraction import (
    extract_confidence_scores,
    extract_publications,
    extract_rich_edge_attributes,
    extract_supporting_text,
)
from TCT.results import KnowledgeGraph

## Sample TRAPI edge attributes

A synthetic edges dict with realistic TRAPI attributes, including nested study results.

In [ ]:
sample_edges = {
    "edge_001": {
        "subject": "NCBIGene:3845",
        "object": "MONDO:0018874",
        "predicate": "biolink:gene_associated_with_condition",
        "sources": [
            {"resource_id": "infores:text-mining-provider-targeted", "resource_role": "primary_knowledge_source"},
            {"resource_id": "infores:biothings-explorer", "resource_role": "aggregator_knowledge_source"},
        ],
        "attributes": [
            {
                "attribute_type_id": "biolink:publications",
                "value": ["PMID:12345678", "PMID:23456789"],
            },
            {
                "attribute_type_id": "biolink:has_supporting_study_result",
                "value": "tmkp:association_001",
                "attributes": [
                    {
                        "attribute_type_id": "biolink:publications",
                        "value": "PMID:34567890",
                    },
                    {
                        "attribute_type_id": "biolink:supporting_text",
                        "value": "KRAS mutations are frequently observed in acute myeloid leukemia.",
                    },
                    {
                        "original_attribute_name": "tmkp_confidence_score",
                        "attribute_type_id": "biolink:has_confidence_level",
                        "value": 0.95,
                    },
                ],
            },
            {
                "attribute_type_id": "biolink:extraction_confidence_score",
                "value": 0.87,
            },
        ],
    },
    "edge_002": {
        "subject": "NCBIGene:7157",
        "object": "MONDO:0018874",
        "predicate": "biolink:gene_associated_with_condition",
        "sources": [
            {"resource_id": "infores:string-db", "resource_role": "primary_knowledge_source"},
        ],
        "attributes": [
            {
                "original_attribute_name": "Combined_score",
                "attribute_type_id": "biolink:has_confidence_level",
                "value": 900,
            },
            {
                "original_attribute_name": "sentences",
                "attribute_type_id": "biolink:description",
                "value": "TP53 is a tumor suppressor commonly mutated in AML.",
            },
        ],
    },
}

## Individual extractors

Each extractor targets a specific attribute type, handling both top-level and nested attributes.

In [ ]:
attrs = sample_edges["edge_001"]["attributes"]

print("Publications:", extract_publications(attrs))
print("Supporting text:", extract_supporting_text(attrs))
print("Confidence scores:", extract_confidence_scores(attrs))

## Combined extraction

`extract_rich_edge_attributes()` runs all extractors in one call.

In [ ]:
extract_rich_edge_attributes(attrs)

## Using with KnowledgeGraph.to_networkx()

Pass `include_attributes=True` to attach extracted metadata directly to graph edges.

In [ ]:
kg = KnowledgeGraph(sample_edges)
G = kg.to_networkx(include_attributes=True)

print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
print()

for u, v, key, data in G.edges(keys=True, data=True):
    print(f"{u} -> {v} ({key})")
    for attr_name, attr_value in data.items():
        print(f"  {attr_name}: {attr_value}")
    print()